# 01 · Carga, limpieza y unión de datos
Proyecto EDA · Campañas de marketing de un banco portugués.
En este notebook: cargamos los datos, detectamos problemas, los limpiamos, unimos las dos tablas y guardamos el resultado.

In [ ]:
import sys
sys.path.append("..")   # para poder importar la carpeta src

import numpy as np
import pandas as pd
from src.funciones import coma_a_numero, fecha_espanola

## 1. Carga de datos

In [ ]:
banco = pd.read_csv("../data/raw/bank-additional.csv")

# El Excel tiene 3 hojas (2012, 2013, 2014): las leemos todas y las juntamos
hojas = pd.read_excel("../data/raw/customer-details.xlsx", sheet_name=None)  # diccionario {nombre: tabla}
lista_tablas = []
for nombre, tabla in hojas.items():
    tabla["anio_hoja"] = int(nombre)      # guardamos de qué hoja viene cada cliente
    lista_tablas.append(tabla)
clientes = pd.concat(lista_tablas, ignore_index=True)

print("Banco:", banco.shape)
print("Clientes:", clientes.shape)

## 2. Primera inspección

In [ ]:
banco.head()

In [ ]:
banco.info()

In [ ]:
# Porcentaje de nulos por columna (solo las que tienen)
nulos = (banco.isna().mean() * 100).round(1)
print(nulos[nulos > 0])

In [ ]:
print("Filas duplicadas:", banco.duplicated().sum())
print("IDs duplicados en banco:", banco["id_"].duplicated().sum())
print("IDs duplicados en clientes:", clientes["ID"].duplicated().sum())
print("Valores de pdays más frecuentes:\n", banco["pdays"].value_counts().head(3))
print("\nValores de 'default':\n", banco["default"].value_counts(dropna=False))

**Problemas detectados**
1. Hay una columna `Unnamed: 0` que sobra (es un índice antiguo) y `latitude`/`longitude` que no vienen en el enunciado.
2. Cuatro columnas numéricas tienen **coma decimal** (`"93,994"`) y se leen como texto.
3. `date` está en español (`2-agosto-2019`) y no existen `contact_month` ni `contact_year`: las crearemos.
4. Hay nulos en `age`, `default`, `euribor3m`, `education`...
5. `pdays = 999` no son 999 días: significa "no contactado antes".
6. `default` casi siempre es 0 (solo 3 unos): no aporta información.

## 3. Limpieza del banco

In [ ]:
# 3.1 Quitar columnas que no sirven
banco = banco.drop(columns=["Unnamed: 0", "latitude", "longitude"])
clientes = clientes.drop(columns=["Unnamed: 0"])

In [ ]:
# 3.2 Columnas con coma decimal -> números
columnas_coma = ["cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]
for col in columnas_coma:
    banco[col] = coma_a_numero(banco[col])

banco[columnas_coma].describe().round(2)

In [ ]:
# 3.3 Fechas en español -> fecha de verdad, y creamos año y mes
banco["date"] = fecha_espanola(banco["date"])
banco["contact_year"] = banco["date"].dt.year
banco["contact_month"] = banco["date"].dt.month
print("Fechas nulas (no se pueden recuperar):", banco["date"].isna().sum())
banco[["date", "contact_year", "contact_month"]].head()

In [ ]:
# 3.4 Variable objetivo y -> 1 (yes) / 0 (no)
banco["y"] = banco["y"].map({"yes": 1, "no": 0})
banco["y"].value_counts()

### 3.5 Nulos de las variables macroeconómicas
Los indicadores macro (`euribor3m`, `cons.price.idx`...) se repiten para todos los clientes de un mismo periodo.
Por eso, en vez de rellenar con la media (que mezclaría épocas muy distintas), rellenamos con la **mediana de las filas que comparten
los mismos valores** de `emp.var.rate`, `cons.conf.idx` y `nr.employed` (estas 3 columnas no tienen nulos).
Comprobé que así `cons.price.idx` se recupera sin error y `euribor3m` con un error medio de 0,02 puntos.

In [ ]:
print("Nulos antes -> euribor3m:", banco["euribor3m"].isna().sum(), "| cons.price.idx:", banco["cons.price.idx"].isna().sum())

grupo = ["emp.var.rate", "cons.conf.idx", "nr.employed"]
for col in ["cons.price.idx", "euribor3m"]:
    mediana_grupo = banco.groupby(grupo)[col].transform("median")
    banco[col] = banco[col].fillna(mediana_grupo)

print("Nulos después -> euribor3m:", banco["euribor3m"].isna().sum(), "| cons.price.idx:", banco["cons.price.idx"].isna().sum())

In [ ]:
# 3.6 Variables categóricas: los nulos pasan a la categoría "unknown" y todo en minúsculas
columnas_texto = ["job", "marital", "education", "contact", "poutcome"]
for col in columnas_texto:
    banco[col] = banco[col].fillna("unknown").str.lower()

In [ ]:
# 3.7 pdays = 999 significa "no contactado antes"
banco["contactado_antes"] = (banco["pdays"] != 999).astype(int)    # 1 = sí, 0 = no
banco["pdays"] = banco["pdays"].replace(999, np.nan)

In [ ]:
# 3.8 Edad: los valores imposibles (menos de 18 o más de 90) pasan a nulo.
# Los nulos de edad (12%) NO se rellenan: inventar la edad de 5.000 personas distorsionaría
# el análisis por edad. Las funciones de pandas ignoran los nulos y simplemente
# usaremos los clientes con edad conocida.
imposibles = (banco["age"] < 18) | (banco["age"] > 90)
print("Edades imposibles:", imposibles.sum())
banco.loc[imposibles, "age"] = np.nan
print("Nulos en age:", banco["age"].isna().sum())

In [ ]:
# 3.9 Outliers: los miramos con el rango intercuartílico (IQR), pero NO los eliminamos
for col in ["duration", "campaign"]:
    q1 = banco[col].quantile(0.25)
    q3 = banco[col].quantile(0.75)
    limite_alto = q3 + 1.5 * (q3 - q1)
    n = (banco[col] > limite_alto).sum()
    print(f"{col}: límite superior = {limite_alto:.0f} -> {n} valores por encima ({n / len(banco):.1%})")
print("Llamadas de duración 0:", (banco["duration"] == 0).sum())

Los outliers (llamadas muy largas o muchos intentos) son valores posibles y además informativos, así que los dejamos.

## 4. Limpieza de los clientes (Excel)

In [ ]:
clientes = clientes.rename(columns={"ID": "id_"})     # mismo nombre que en el banco para unir
clientes["Dt_Customer"] = pd.to_datetime(clientes["Dt_Customer"])
clientes["hijos_total"] = clientes["Kidhome"] + clientes["Teenhome"]
print("Nulos en clientes:", clientes.isna().sum().sum())
clientes.describe().round(1)

## 5. Unión de las dos tablas
Unimos por `id_` con un **left join** desde el banco: el análisis trata sobre las llamadas, así que queremos conservarlas todas.

In [ ]:
union = banco.merge(clientes, on="id_", how="left", indicator=True)
print(union["_merge"].value_counts())
print("Clientes del Excel que no están en el banco:", (~clientes["id_"].isin(banco["id_"])).sum())

In [ ]:
df = union.drop(columns="_merge")

## 6. Nuevas variables

In [ ]:
# Grupos de edad
df["grupo_edad"] = pd.cut(df["age"], bins=[17, 25, 35, 45, 55, 65, 91],
                          labels=["18-25", "26-35", "36-45", "46-55", "56-65", "66+"])

# Cuartiles de ingresos
df["tramo_ingresos"] = pd.qcut(df["Income"], q=4, labels=["Q1 (bajo)", "Q2", "Q3", "Q4 (alto)"])

# Años desde que es cliente hasta la llamada
df["antiguedad_anios"] = ((df["date"] - df["Dt_Customer"]).dt.days / 365).round(2)

df[["age", "grupo_edad", "Income", "tramo_ingresos", "antiguedad_anios", "hijos_total"]].head()

## 7. Guardado

In [ ]:
df.to_csv("../data/processed/bank_marketing_clean.csv", index=False)
print("Dataset final:", df.shape)
df.info()